# `select_points_in_box()`

`nematics3d.geometry.select_points_in_box()` selects 3D points that lie inside or on an oriented rectangular box. The box may be translated and rotated; it does not need to be aligned with the coordinate axes.

## Box convention

The box is specified by an array `corners` with at least four rows. The ordering of the first four rows matters:

- `corners[0]` is one reference corner.
- `corners[1] - corners[0]`, `corners[2] - corners[0]`, and `corners[3] - corners[0]` are the three box edges leaving that corner.
- These three edges must have positive length and be mutually perpendicular.

This is the same first-four-corner convention used by `get_box_corners()`. Additional rows are accepted but are not needed by `select_points_in_box()`.

## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The following cell only imports `NumPy` and the box geometry helpers.

In [ ]:
import numpy as np

from nematics3d.geometry import get_box_corners, select_points_in_box

## Minimal example

First create a box spanning $0 \le x \le 2$, $0 \le y \le 1$, and $0 \le z \le 1$, then select points inside or on its boundary.

In [ ]:
corners = get_box_corners(2.0, 1.0, 1.0)
points = np.array([
    [0.5, 0.5, 0.5],   # inside
    [2.0, 0.5, 0.5],   # on a face
    [2.2, 0.5, 0.5],   # outside
    [-0.1, 0.0, 0.0],  # outside
])

selected = select_points_in_box(points, corners)
print(selected)

## Returning the selection mask

Set `is_return_mask=True` when the same selection must also be applied to another array whose rows correspond to `points`. The mask always refers to the original input-row order.

In [ ]:
selected, mask = select_points_in_box(
    points,
    corners,
    is_return_mask=True,
)

values = np.array([10.0, 20.0, 30.0, 40.0])
print("mask:", mask)
print("selected points:\n", selected)
print("corresponding values:", values[mask])

## Rotated and translated boxes

The function is not restricted to axis-aligned boxes. Here the first edge points along $(1, 1, 0)$, the second along $(-1, 1, 0)$, and the third along the $z$ axis. These directions are mutually perpendicular.

In [ ]:
origin = np.array([3.0, -1.0, 2.0])
axis1 = np.array([1.0, 1.0, 0.0]) / np.sqrt(2.0)
axis2 = np.array([-1.0, 1.0, 0.0]) / np.sqrt(2.0)
axis3 = np.array([0.0, 0.0, 1.0])

corners_rotated = np.array([
    origin,
    origin + 2.0 * axis1,
    origin + 1.0 * axis2,
    origin + 0.5 * axis3,
])

points_rotated = np.array([
    origin + 1.0 * axis1 + 0.5 * axis2 + 0.25 * axis3,
    origin + 2.2 * axis1 + 0.5 * axis2 + 0.25 * axis3,
])

print(select_points_in_box(points_rotated, corners_rotated))

## Arguments and special behavior

The public signature is:

```python
select_points_in_box(points, corners, is_return_mask=False, *, atol=1e-9)
```

| Argument | Meaning |
| --- | --- |
| `points` | Finite 3D points with shape `(N, 3)`. Empty input is allowed. |
| `corners` | At least four finite 3D corners following the convention above. `None` means no box filtering. |
| `is_return_mask` | If `True`, also return the boolean mask over the original points. |
| `atol` | Non-negative absolute tolerance applied at the six box faces. |

When `corners=None`, every input point is selected. This is useful when a caller treats box clipping as optional but wants to keep one code path.

### Boundary tolerance

Points exactly on a box face are included. `atol` also permits small absolute coordinate deviations beyond a face, which is useful for floating-point geometry. Because this tolerance is absolute rather than relative, choose it in the same physical units as the point coordinates.

In [ ]:
near_boundary = np.array([
    [2.0, 0.5, 0.5],
    [2.0 + 5e-10, 0.5, 0.5],
    [2.0 + 2e-9, 0.5, 0.5],
])

_, mask = select_points_in_box(
    near_boundary,
    corners,
    is_return_mask=True,
)
print(mask)

## How the geometric test works

Let $\mathbf{c}_0$ be the reference corner and let the three normalized edge directions be $\hat{\mathbf{e}}_1$, $\hat{\mathbf{e}}_2$, and $\hat{\mathbf{e}}_3$, with corresponding edge lengths $L_1$, $L_2$, and $L_3$. For each point $\mathbf{x}$, the function computes local box coordinates

$$u_i = (\mathbf{x} - \mathbf{c}_0) \cdot \hat{\mathbf{e}}_i. $$

The point is selected when all three coordinates satisfy

$$-\mathrm{atol} \le u_i \le L_i + \mathrm{atol}. $$

Before applying this test, the function verifies that the three defining edges have positive length and are mutually perpendicular. A skew parallelepiped is therefore rejected instead of being silently interpreted as a rectangular box.

## Summary

- `select_points_in_box()` selects finite 3D points inside or on a rectangular box.
- The box may be translated and rotated, but its three defining edges must be mutually perpendicular.
- The first four rows of `corners` define the reference corner and three outgoing edges.
- Set `is_return_mask=True` when the selection must be reused for corresponding data.
- `atol` controls absolute floating-point tolerance at the box faces.
- `corners=None` disables box filtering and selects every input point.